# A1 - Inference timing only (CPU or CUDA)

Notebook cũ chỉ đo GPU nên `inference_ms_per_image_mean` là null khi chạy CPU.
Notebook này đo trực tiếp **CPU** mặc định, hoặc CUDA nếu chủ động chọn.
Không chạy lại training, validation, official test, prediction export hay gallery.

Đọc checkpoint gốc và profile đã đánh giá `cpu-0bd6ba8e08e9` trên Drive.
Đo cùng batch 64, FP32, 20 warm-up và 100 forwards/run; lưu raw timing samples.
Summary mới giữ nguyên các quality metrics và bổ sung thời gian với nhãn device/environment.
Profile/summary cũ có hash vẫn được giữ nguyên.

Chọn CPU runtime nếu hết quota GPU, rồi **Runtime → Run all**.


In [ ]:
from pathlib import Path
import os

ARTIFACT_ROOT=Path('/content/drive/MyDrive/dlbench_a1_main')
EVALUATION_PROFILE=ARTIFACT_ROOT/'postprocess_v1/cpu-0bd6ba8e08e9'
TIMING_ROOT=ARTIFACT_ROOT/'inference_timing_v1'
CHECKOUT=Path('/content/deep-learning-benchmark-suite')
REPO_URL='https://github.com/TuanKhai1210/Deep-Learning-Benchmark-Suite.git'
TIMING_DEVICE='cpu' # Explicit CPU measurement. Change to 'cuda' only with a GPU runtime.
CPU_THREADS=2
AUTO_DOWNLOAD_ZIP=True
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')


## 1. Mount Drive and restore the recorded source


In [ ]:
import json
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')
ledger_path = ARTIFACT_ROOT / 'run_ledger.json'
if not ledger_path.is_file():
    raise FileNotFoundError(f'Không thấy {ledger_path}. Kiểm tra ARTIFACT_ROOT.')
training_ledger = json.loads(ledger_path.read_text())
pinned_commit = training_ledger['identity']['git_commit']
if len(pinned_commit) != 40 or any(c not in '0123456789abcdef' for c in pinned_commit.lower()):
    raise ValueError('Invalid ledger commit')
if not CHECKOUT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(CHECKOUT)], check=True)
if not (CHECKOUT / '.git').is_dir():
    raise RuntimeError(f'Not a Git checkout: {CHECKOUT}')

def source_git(*args):
    return subprocess.run(['git', *args], cwd=CHECKOUT, check=True,
                          capture_output=True, text=True).stdout.strip()

if source_git('status', '--porcelain'):
    raise RuntimeError('Source checkout has local changes; preserve them and use a fresh runtime.')
try:
    source_git('cat-file', '-e', pinned_commit + '^{commit}')
except subprocess.CalledProcessError:
    subprocess.run(['git', 'fetch', 'origin', pinned_commit], cwd=CHECKOUT, check=True)
if source_git('rev-parse', 'HEAD') != pinned_commit and any(
    name == 'dlbench' or name.startswith('dlbench.') for name in sys.modules):
    raise RuntimeError('Old dlbench is already imported. Restart the runtime before changing revision.')
subprocess.run(['git', 'checkout', '--detach', pinned_commit], cwd=CHECKOUT, check=True)
os.chdir(CHECKOUT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[ml]'], check=True)
sys.path.insert(0, str(CHECKOUT / 'src'))
if source_git('rev-parse', 'HEAD') != pinned_commit or source_git('status', '--porcelain'):
    raise RuntimeError('Source revision/clean-state gate failed')
print('Pinned source:', pinned_commit)


## 2. Timing-only helpers
No evaluation/training APIs are invoked.


In [ ]:
"""Measure CPU/CUDA inference and derive a new summary; no data evaluation."""

from pathlib import Path
from copy import deepcopy
from statistics import mean, median, stdev
from uuid import uuid4
import csv
import hashlib
import json
import math
import os
import zipfile

TIMING_VERSION = 'a1-timing-only-20261024-v1'
TIMING_MODELS = ('linear', 'mlp', 'cnn', 'rnn', 'transformer')
TIMING_SEEDS = (69420, 67, 69)


def timing_read(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def timing_save(path, record):
    Path(path).write_text(json.dumps(record, indent=2, ensure_ascii=False, allow_nan=False)+'\n', encoding='utf-8')


def timing_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()


def timing_digest(record):
    return hashlib.sha256(json.dumps(record, sort_keys=True, separators=(',', ':'), allow_nan=False).encode()).hexdigest()


def timing_stage(destination, identity, action):
    destination = Path(destination)
    if destination.exists():
        marker = timing_read(destination / '_complete.json')
        if marker['identity'] != identity:
            raise RuntimeError('Timing cache identity differs: '+str(destination))
        for relative, expected in marker['files'].items():
            path = (destination/relative).resolve()
            if not path.is_relative_to(destination.resolve()) or timing_hash(path) != expected:
                raise RuntimeError('Timing cache hash differs: '+str(path))
        print('SKIP verified timing:', destination.name, flush=True)
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name('.'+destination.name+'.partial-'+uuid4().hex[:10])
    temporary.mkdir()
    action(temporary)
    files = {p.relative_to(temporary).as_posix():timing_hash(p) for p in temporary.rglob('*') if p.is_file()}
    timing_save(temporary/'_complete.json', {'identity':identity,'files':files})
    temporary.rename(destination)


def derive_timed_summary(source_profile, destination, records, device, timing_profile):
    """Replace timing fields only; preserve all saved quality metrics exactly."""
    source_profile, destination = Path(source_profile), Path(destination)
    aggregates = timing_read(source_profile/'summary/mean_std.json')
    with (source_profile/'summary/per_run.csv').open(encoding='utf-8', newline='') as stream:
        rows = list(csv.DictReader(stream))
    if len(rows) != 30 or len(aggregates) != 10:
        raise ValueError('Expected 30 saved evaluation rows and ten aggregate rows')
    by_run = {record['run_id']:record for record in records}
    expected = {(model,seed) for model in TIMING_MODELS for seed in TIMING_SEEDS}
    if {(r['model'],r['seed']) for r in records} != expected or len(records) != 15:
        raise ValueError('Timing requires all 15 model/seed pairs')
    for row in rows:
        record = by_run[row['run_id']]
        if int(row['parameter_count']) != record['measurement']['parameter_count']:
            raise ValueError('Parameter count differs')
        row.update(inference_ms_per_image=record['measurement']['amortized_ms_per_image'],
            inference_device=device, inference_status='completed', inference_timing_profile=timing_profile)
    for aggregate in aggregates:
        group = [r for r in records if r['model']==aggregate['model']]
        values = [r['measurement']['amortized_ms_per_image'] for r in group]
        aggregate.update(inference_ms_per_image_mean=mean(values),
            inference_ms_per_image_sample_sd=stdev(values),
            inference_device=device, inference_timing_profile=timing_profile)
    timing_save(destination/'mean_std.json', aggregates)
    for name, data in [('mean_std.csv',aggregates),('per_run.csv',rows)]:
        with (destination/name).open('w',encoding='utf-8',newline='') as stream:
            writer=csv.DictWriter(stream,fieldnames=list(data[0]));writer.writeheader();writer.writerows(data)


def run_timing_only(artifact_root, evaluation_profile, output_root, device='cpu', cpu_threads=2):
    import torch
    from dlbench.a1.benchmark import benchmark_inference
    from dlbench.a1.checkpoint import load_checkpoint
    from dlbench.a1.models.registry import build_model
    from dlbench.common.reproducibility import collect_environment, seed_everything
    artifact_root, evaluation_profile = Path(artifact_root), Path(evaluation_profile)
    ledger = timing_read(artifact_root/'run_ledger.json')
    source_session = timing_read(evaluation_profile/'session.json')
    if source_session['source_ledger'] != ledger['identity']:
        raise RuntimeError('Evaluation and checkpoint ledgers differ')
    marker = timing_read(evaluation_profile/'summary/_complete.json')
    if marker['identity']['profile'] != source_session['identity']:
        raise RuntimeError('Summary profile differs')
    for relative, expected_hash in marker['files'].items():
        path = (evaluation_profile/'summary'/relative).resolve()
        if not path.is_relative_to((evaluation_profile/'summary').resolve()) or timing_hash(path) != expected_hash:
            raise RuntimeError('Existing evaluation summary hash differs')
    actual_device = ('cuda' if torch.cuda.is_available() else 'cpu') if device=='auto' else device
    if actual_device not in ('cpu','cuda') or (actual_device=='cuda' and not torch.cuda.is_available()):
        raise RuntimeError('Requested timing device unavailable')
    if actual_device=='cpu':
        torch.set_num_threads(int(cpu_threads))
    seed_everything(TIMING_SEEDS[0])
    torch.set_float32_matmul_precision('highest')
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=True
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=False
    environment=collect_environment()
    identity={'version':TIMING_VERSION, 'code_sha256':globals().get('TIMING_CODE_SHA256',TIMING_VERSION),
        'device':actual_device, 'cpu_threads':torch.get_num_threads(),
        'environment':{k:v for k,v in environment.items() if k!='system_ram'},
        'source_evaluation_profile':evaluation_profile.name,
        'source_summary_sha256':timing_hash(evaluation_profile/'summary/mean_std.json')}
    output=Path(output_root)/(actual_device+'-'+timing_digest(identity)[:12])
    output.mkdir(parents=True,exist_ok=True)
    if not (output/'session.json').exists():
        timing_save(output/'session.json', {'identity':identity,'environment':environment,
            'source_ledger':ledger['identity'],'source_evaluation_profile':str(evaluation_profile),
            'training_performed':False,'validation_or_test_rerun':False})
    records=[]
    for model_name in TIMING_MODELS:
        for seed in TIMING_SEEDS:
            run_name=ledger['completed'][f'{model_name}:{seed}']
            if Path(run_name).name!=run_name:
                raise ValueError('Invalid run ID')
            run=artifact_root/'main'/run_name
            config=timing_read(run/'config.json')
            checkpoint=run/'best.pt'
            checkpoint_hash=timing_hash(checkpoint)
            eval_metrics=timing_read(evaluation_profile/run_name/'test/metrics.json')
            if (config['model']['name'],config['run']['seed'])!=(model_name,seed) or eval_metrics['checkpoint_sha256']!=checkpoint_hash:
                raise RuntimeError('Checkpoint/config differs from the evaluated run')
            timing=deepcopy(config['timing'])
            timing['device']=actual_device  # Explicit CPU protocol variant, never relabeled CUDA.
            context={'session':identity,'run_id':run_name,'checkpoint_sha256':checkpoint_hash,
                     'config_sha256':timing_hash(run/'config.json'),'effective_timing':timing}
            def measure(path):
                seed_everything(seed)
                payload=load_checkpoint(checkpoint,map_location=actual_device)
                if payload['config']['model']!=config['model'] or payload['run_seed']!=seed:
                    raise RuntimeError('Checkpoint architecture/seed differs')
                architecture=build_model(config['model']).to(actual_device)
                architecture.load_state_dict(payload['model_state_dict'])
                shape=[timing['batch_size'],config['preprocessing']['channels'],*config['preprocessing']['image_size']]
                batch=torch.zeros(shape,dtype=torch.float32,device=actual_device)
                measured=benchmark_inference(architecture,batch,timing)
                record={'run_id':run_name,'model':model_name,'seed':seed,'status':'completed',
                    'checkpoint_sha256':checkpoint_hash,'requested_training_timing':config['timing'],
                    'effective_timing':timing,'measurement':measured,'environment':environment,
                    'input_policy':'Preallocated zeros, forward-only; excludes loading, transfer, checkpoint I/O'}
                timing_save(path/'benchmark.json',record)
                del architecture,payload,batch
                if actual_device=='cuda':torch.cuda.empty_cache()
            stage=output/'runs'/run_name
            timing_stage(stage,context,measure)
            record=timing_read(stage/'benchmark.json')
            measured=record['measurement']
            samples=measured['batch_ms_samples']
            if len(samples)!=timing['measurement_steps'] or any(not math.isfinite(t) or t<=0 for t in samples):
                raise RuntimeError('Invalid timing samples')
            if not math.isclose(median(samples)/timing['batch_size'],measured['amortized_ms_per_image'],abs_tol=1e-12):
                raise RuntimeError('Timing aggregation differs')
            records.append(record)
            print(model_name,seed,actual_device,f'{measured["amortized_ms_per_image"]:.6f} ms/image',flush=True)
    summary_context={'session':identity,'benchmarks':{
        r['run_id']:timing_hash(output/'runs'/r['run_id']/'benchmark.json') for r in records}}
    timing_stage(output/'summary',summary_context,
        lambda path:derive_timed_summary(evaluation_profile,path,records,actual_device,output.name))
    archive=output.with_suffix('.zip')
    if not archive.exists():
        temporary=archive.with_suffix('.partial.zip')
        with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as z:
            for path in sorted(output.rglob('*')):
                if path.is_file() and not any(p.startswith('.') for p in path.relative_to(output).parts):
                    z.write(path,output.name+'/'+path.relative_to(output).as_posix())
        temporary.replace(archive)
    print('\nDONE. New summary:',output/'summary/mean_std.json',flush=True)
    print('Measured device:',actual_device,'| ZIP:',archive,flush=True)
    print('Batch-amortized ms/image at batch 64; not batch-one latency.',flush=True)
    return output,archive

TIMING_CODE_SHA256 = '45002f1fce51eccd68163d6999eddfae051093e691f6f217ae30fdc6d90cf170'


## 3. Measure all 15 checkpoints and derive the new summary

The metric is median batch milliseconds divided by batch size, then averaged over the three saved seeds.
It is **not** batch-one latency. CPU results must be labeled CPU; they do not replace T4/CUDA timings.
Saved per-run timings can be reused after their identity and file hashes are verified.


In [ ]:
if source_git('rev-parse','HEAD')!=pinned_commit or source_git('status','--porcelain'):
    raise RuntimeError('Source checkout changed')
TIMING_OUTPUT,TIMING_ZIP=run_timing_only(
    ARTIFACT_ROOT,EVALUATION_PROFILE,TIMING_ROOT,
    device=TIMING_DEVICE,cpu_threads=CPU_THREADS)


## 4. Results
The ZIP contains new timing records and a new summary. Original test/analysis output remains unchanged.


In [ ]:
for row in timing_read(TIMING_OUTPUT/'summary/mean_std.json'):
    if row['eval_split']=='test':
        print(row['model'],row['inference_device'],
              f"{row['inference_ms_per_image_mean']:.6f} ± {row['inference_ms_per_image_sample_sd']:.6f} ms/image")
print('New summary:',TIMING_OUTPUT/'summary/mean_std.json')
if AUTO_DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(TIMING_ZIP))
